### Decorator ohne Argumente
Ein **Decorator** ist eine Funktion, die eine andere Funktion modifiziert,
d.h. eine **Funktion als Argument nimmt**, und eine **Funktion zurück gibt**.  

```python
@foo
def f(x):
    ...
```
ist **syntactic  sugar** für

```python
def f(x):
    ...
    
f = foo(f)    
```

**Typische Anwendung**:
Im Modul `grid_helpers.py` haben wir viele Funktionen, die Etwas auf 
ein Leinwandobjekt `canvas` zeichen und dabei eines der Attribute `fill_style`, `stroke_style`, `line_width`, ...
modifizieren.

```python
def draw_gird(canvas, grid_spec, color='black'):
    canvas.stroke_style = color
    # zeichne das Gitter
```

Es wäre besser, wenn die Funktion `draw_gird` das Attribut `stroke_style` am Ende wieder auf
den ursprünglichen Wert zurücksetzen würde:

```python
def draw_gird(canvas, grid_spec, color='black'):
    canvas.save()     # speichert alle wesentlichen Attribute
    canvas.stroke_style = color
    # zeichne das Gitter
    canvas.restore()  # setzt Attribute wieder zurueck
```

Statt jede Funktion manuell zu editieren, scheiben wir eine Funktion, welche
die bestehenden Funktionen entsprechend anpassen.  
**Beachte**: `save(f)` nimmt eine Funktion als Argument und gibt eine Funktion zurück!

```python
def save(f):
    def wrapper(canvas, *args, **kwargs):
        canvas.save()
        f(canvas, *args, **kwargs)  # rufe f mit den erhaltenen Args auf
        canvas.restore()

    return wrapper
```


Eine Variante von `draw_grid`, welche die Attribute nicht ändert erhalten wir dann so:  
```python
    draw_gird =  save(draw_grid)
```
Jede Funktion, die Attribute modifiziert (ohne Rückgabewert), kann man nun so
in eine sicheren Variante umwandeln:
```python
@save
def draw_gird(canvas, grid_spec, color='black'):
    canvas.stroke_style = color
    # zeichne das Gitter
```


**Aufgabe**:  
Füge den Decorator `save` als erste Funktion ins Modul `grid_helpers` ein und dekoriere dann
alle Funktionen, die Canvas-Attribute modifizieren.

***
Beispiel: `show_args` ist ein Decorator, der
ausgibt, mit welchen Argumenten eine Funktion aufgerufen wurde, und dann die
Funktion mit diesen Argumenten aufruft und das Resultat zurück gibt.
***

In [ ]:
def show_args(f):
    def wrapper(*args, **kwargs):
        # Argumente  ausgeben
        print(f'Die Funktion "{f.__name__}" wurde mit folgenden Argumentn aufgerufen:')
        print(f'    Positionale Argumente: {args}')
        print(f'    Keyword Argumente: {kwargs}')

        # nun originale f aufrufen und Resultat zurueckgeben
        return f(*args, **kwargs)

    return wrapper

In [ ]:
def insert(s, i, t, lower=False):
    if lower:
        t = t.lower()
    return s[:i] + t + s[i:]

In [ ]:
insert('testtest', 4, 'FOO')

In [ ]:
g = show_args(insert)
g('testtest', 4, 'FOO', lower=True)

In [ ]:
g.__name__  # aber der Name ist 'wrapper'!

In [ ]:
insert = show_args(insert)
insert('testtest', 4, 'FOO', lower=True)

In [ ]:
insert.__name__  # der Name ist  'wrapper'

In [ ]:
@show_args
def insert_1(s, i, t, lower=False):
    if lower:
        t = t.lower()
    return s[:i] + t + s[i:]

In [ ]:
insert_1('testtest', 4, 'FOO')

In [ ]:
insert.__name__

***
Das Attribut `wrapper.__name__`  kann natürlich leicht korrigiert werden.  
Es sind aber noch weitere Attribute zu korrigieren.  
Python stellt deshalb einen Decorator zur Verfügung der dies tut.
***

In [ ]:
def show_args_fix_name(f):
    def wrapper(*args, **kwargs):
        # Argumente  ausgeben
        print(f'Die Funktion "{f.__name__}" wurde mit folgenden Argumentn aufgerufen:')
        print(f'    Positionale Argumente: {args}')
        print(f'    Keyword Argumente: {kwargs}')

        # nun f aufrufen und Resultat zurueckgeben
        return f(*args, **kwargs)

    wrapper.__name__ = f.__name__
    return wrapper

In [ ]:
@show_args_fix_name
def insert_2(s, i, t, lower=False):
    if lower:
        t = t.lower()
    return s[:i] + t + s[i:]

In [ ]:
insert_2.__name__  # name nun richtig.

In [ ]:
insert_2.__qualname__  # name nun richtig.

***
Typische Form eines Decorators
***

In [ ]:
from functools import wraps


def show_args_1(f):
    @wraps(f)  # korrigiert  den Namen und alle weitere Attribute von wrapper
    def wrapper(*args, **kwargs):
        # Argumente  ausgeben
        print(f'Die Funktion "{f.__name__}" wurde mit folgenden Argumentn aufgerufen:')
        print(f'    Positionale Argumente: {args}')
        print(f'    Keyword Argumente: {kwargs}')

        # nun f aufrufen und Resultat zurueckgeben
        return f(*args, **kwargs)

    return wrapper

In [ ]:
@show_args_1
def test(s):
    print(s)

In [ ]:
test.__name__, test.__qualname__

In [ ]:
test('foo')